In [ ]:
abc = 'abc'
xyz = 'xyz'
n_jobs = 1
iter_cv = 100
nrand = 10

In [1]:
pip install neureval==1.1.1

,Collecting neureval
,  Downloading neureval-0.1.0.tar.gz (33 kB)
,Requirement already satisfied: numpy in /opt/conda/lib/python3.6/site-packages (from neureval) (1.18.5)
,Requirement already satisfied: scipy in /opt/conda/lib/python3.6/site-packages (from neureval) (1.5.4)
,Collecting scikit-learn
,  Downloading scikit_learn-0.24.2-cp36-cp36m-manylinux2010_x86_64.whl (22.2 MB)
,     |████████████████████████████████| 22.2 MB 87.7 MB/s eta 0:00:01     |████████████████████████        | 16.6 MB 87.7 MB/s eta 0:00:01
,Collecting umap-learn
,  Downloading umap-learn-0.5.3.tar.gz (88 kB)
,     |████████████████████████████████| 88 kB 92.9 MB/s eta 0:00:01
,Requirement already satisfied: matplotlib in /opt/conda/lib/python3.6/site-packages (from neureval) (3.0.3)
,Requirement already satisfied: cycler>=0.10 in /opt/conda/lib/python3.6/site-packages (from matplotlib->neureval) (0.11.0)
,Requirement already satisfied: kiwisolver>=1.0.1 in /opt/conda/lib/python3.6/site-packages (from matplotli

In [3]:
pip install mvlearn

In [3]:
%%bash
dx cd /prove_cohorts/prova_clustering_MDD_FAST_DKT_FA/neureval_MDD_RDS-4dep/new_analysis/DKT_cortical_thickness_only/training_55_test_45

In [4]:
%%bash
dx download data_55_training_DKT_thickness.csv

In [5]:
%%bash
dx download covariates_55_training_DKT_thickness.csv

In [6]:
# Make the required imports
import pandas as pd
import numpy as np
from neureval.param_selection_confounds import ParamSelectionConfounds
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import AgglomerativeClustering, KMeans, DBSCAN, HDBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
import umap
import os
import pickle as pkl
import sys
from neureval.utils import kuhn_munkres_algorithm
import logging

In [7]:
# Define working directories and create a new folder called 'models'
main_path = '/opt/notebooks'
data_path = main_path
os.chdir(data_path)

In [8]:
# Define output folder called 'results'. 
# Within 'results, create a subfolder corresponding to the input features (e.g., 'GM + FA').
# Within this subfolder, create another folder corresponding to the set of covariates used (e.g., 'age_sex_TIV')
#out_dir = os.path.join(data_path,'results', 'prova_jupyter')
#os.makedirs(out_dir, exist_ok=True)
#os.chdir(out_dir)

In [9]:
# Import data and covariates files
#database = 'Participant_table_MDD_FAST_50.csv'
#covariates_file = 'Covariates_MDD_FAST_50.csv'
# If there are multiple sheets, specify the name of the current sheet 
data = pd.read_csv('/opt/notebooks/data_55_training_DKT_thickness.csv', delimiter=';')
cov = pd.read_csv('/opt/notebooks/covariates_55_training_DKT_thickness.csv', delimiter=';')

In [ ]:
# Define two dictionaries for modalities (i.e., views in multiview clustering) and covariates variables.
# For each kind of modality, specify the indexes of the columuns related to the features to be used for clustering
# You can also specify different covaiates for each kind of modality
modalities = {'gm_thickness': data.iloc[:,1:]}
covariates = {'gm_thickness': cov.iloc[:,1:-1]}

In [12]:
# Define multiview clustering and classifier parameters to be optimized.
# params should be a dictionary of the form {‘s’: {classifier parameter grid}, ‘c’: {clustering parameter grid}} 
# including the lists of classifiers and clustering methods to fit to the data.
params = {'s': {'C': [0.01, 0.1, 1, 10, 100],
                'kernel':['linear']},
          'c': {'init':['k-means++']}}

In [13]:
# Specify clustering (c), classifier (s), and preprocessing (preproc) algorithms
c = KMeans(random_state=42)
s = SVC(random_state=42)

In [ ]:
# Parameters selection
best_model = ParamSelectionConfounds(params, 
                                      cv=2, 
                                      s=s, 
                                      c=c,
                                      preprocessing=None,
                                      nrand=nrand,
                                      n_jobs=n_jobs,
                                      iter_cv=iter_cv,
                                      clust_range=list(range(2,5)),
                                      strat=None)

best_model.fit(data,modalities,covariates)

# Save model's parameters in the output directory. Change file name to match the model you performed
best_results = best_model.best_param_
pkl.dump(best_results, open('./best_results_training_55_KMeans_SVM.pkl', 'wb'))

12:13:36, INFO Running 12 combinations of parameters...
,
